# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**Unit of analysis.** One row represents one content item for one client (`client_hash_id × content_hash_id`). The source table `fact_content_daily_performance` is one row per client × content × day; we aggregate the daily rows within separate feature and label windows.

**Two windows, deliberately separated.**

| Window | Dates | Role |
|---|---|---|
| February 2026 | 2026-02-01 → 2026-02-28 | Features — information knowable by February 28 |
| March 2026 | 2026-03-01 → 2026-03-31 | Label/outcome — what happens after the feature window |

The windows do not overlap. This prevents information from the outcome period leaking into the features.

**Target.** `went_dark = 1` when a content item has zero measured GSC clicks during March 2026. This represents a content item losing its click activity after the February feature window.

**Deliberate exclusion.** March outcome fields and label-derived fields are excluded from the feature set because they would not be knowable at the February 28 decision moment.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import numpy as np
from google.colab import userdata

# Get token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Store token as a DuckDB session variable rather than putting it in SQL text
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# Separate feature and label windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected successfully.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected successfully.
Feature window: February 2026
Label window: March 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features
The feature set will use February information that is available by 2026-02-28, such as prior GSC performance and content characteristics.

### Label
`went_dark` — 1 when measured March GSC clicks are zero, otherwise 0.

### Context
`client_hash_id`, `content_hash_id`, and `report_date` are used for grouping, joining, filtering, and time-based splitting. They are not model features.

### Excluded
March outcome variables and label-derived fields are excluded because they contain future information relative to the February 28 decision point. In particular, `trend_direction`, `trend_pct`, and `is_declining_label` must not be used as features.

In [20]:
# Inspect the February 2026 warehouse schema
schema = con.sql(f"""
DESCRIBE SELECT *
FROM {FEB}
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain verification

The source fact table should contain at most one row for each `report_date × client_hash_id × content_hash_id` combination.

In [21]:
# Query 1 — Verify source grain

grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {FEB}
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

display(grain_check)

print("Duplicate groups found:", len(grain_check))

,report_date,client_hash_id,content_hash_id,row_count


Duplicate groups found: 0


Query 2 — February row count + date span



In [22]:
# Query 2 — February row count and date span

coverage = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FEB}
""").df()

display(coverage)

,row_count,min_date,max_date
0,7355108,2026-02-01,2026-02-28


Availability using IS TRUE

Add another code cell:

In [23]:
# Query 3 — GSC availability

availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM {FEB}
""").df()

display(availability)

,total_rows,gsc_available_rows
0,7355108,2621783


build your five-feature frame

After those three queries, we'll create the actual feature dataframe.

In [24]:
# Build the February feature frame
# Keep availability separate from measured values.

features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS feb_gsc_impressions,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS feb_gsc_clicks,

    AVG(gsc_avg_position) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS feb_gsc_avg_position,

    SUM(sessions_organic) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS feb_organic_sessions,

    SUM(ga4_pageviews) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS feb_pageviews,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS feb_gsc_measured_days,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS feb_ga4_measured_days

FROM {FEB}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

display(features.head())
print("Feature frame shape:", features.shape)

,client_hash_id,content_hash_id,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_avg_position,feb_organic_sessions,feb_pageviews,feb_gsc_measured_days,feb_ga4_measured_days
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,NaN,NaN,28,0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,3.0,6.0,28,5
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.0,1.0,28,1
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,5.0,9.0,28,4
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1.0,3.0,28,3


Feature frame shape: (321546, 9)


In [25]:
# Cell 5 — Build the March 2026 outcome label

label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS mar_gsc_clicks,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS mar_gsc_measured_days

FROM {MAR}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# Join the March outcome to the February feature frame
frame = features.merge(
    label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# A label is only defined when March had at least one measured GSC day
frame = frame[frame["mar_gsc_measured_days"].fillna(0) > 0].copy()

# Create the outcome label
frame["went_dark"] = (
    frame["mar_gsc_clicks"] == 0
).astype(int)

print("Final frame shape:", frame.shape)
print("Label distribution:")
print(frame["went_dark"].value_counts())
print("\nLabel rate:", frame["went_dark"].mean())

display(frame.head())

Final frame shape: (161539, 12)
Label distribution:
went_dark
1    98368
0    63171
Name: count, dtype: int64

Label rate: 0.608942732095655


,client_hash_id,content_hash_id,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_avg_position,feb_organic_sessions,feb_pageviews,feb_gsc_measured_days,feb_ga4_measured_days,mar_gsc_clicks,mar_gsc_measured_days,went_dark
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,NaN,NaN,28,0,0.0,29.0,1
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,3.0,6.0,28,5,4.0,29.0,0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.0,1.0,28,1,0.0,29.0,1
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,5.0,9.0,28,4,5.0,29.0,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1.0,3.0,28,3,1.0,29.0,0


In [26]:
# Cell 6 — Deliberate label leakage experiment

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Exactly five honest features
honest_features = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_organic_sessions",
    "feb_pageviews"
]

# Keep only rows with a defined label
model_data = frame.dropna(subset=["went_dark"]).copy()

X = model_data[honest_features]
y = model_data["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Simple model with median imputation for missing feature values
honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)
honest_prob = honest_model.predict_proba(X_test)[:, 1]

honest_accuracy = accuracy_score(y_test, honest_pred)
honest_auc = roc_auc_score(y_test, honest_prob)

print("HONEST MODEL")
print("Accuracy:", round(honest_accuracy, 4))
print("ROC-AUC:", round(honest_auc, 4))


# ---------------------------------------------------------
# DELIBERATE LEAK
# ---------------------------------------------------------

leaky_features = honest_features + ["went_dark"]

X_leak = model_data[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

leaky_model.fit(X_train_l, y_train_l)

leaky_pred = leaky_model.predict(X_test_l)
leaky_prob = leaky_model.predict_proba(X_test_l)[:, 1]

leaky_accuracy = accuracy_score(y_test_l, leaky_pred)
leaky_auc = roc_auc_score(y_test_l, leaky_prob)

print("\nDELIBERATE LEAK")
print("Accuracy:", round(leaky_accuracy, 4))
print("ROC-AUC:", round(leaky_auc, 4))

print("\nScore jump:")
print("Accuracy change:", round(leaky_accuracy - honest_accuracy, 4))
print("ROC-AUC change:", round(leaky_auc - honest_auc, 4))

HONEST MODEL
Accuracy: 0.825
ROC-AUC: 0.8697

DELIBERATE LEAK
Accuracy: 1.0
ROC-AUC: 1.0

Score jump:
Accuracy change: 0.175
ROC-AUC change: 0.1303


### Leakage lesson

I deliberately added the label `went_dark` as a feature. The quick model score jumped toward a perfect score because the model was given the answer directly. This is label leakage: `went_dark` is determined from the March outcome and would not be known at the February 28 decision moment.

I removed the leaked column and kept the honest five-feature result. The final feature set contains only information available before the March outcome window.

In [27]:
# Cell 7 — Final honest feature set after removing leakage

final_features = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_organic_sessions",
    "feb_pageviews"
]

print("Final features after leakage removal:")
for i, feature in enumerate(final_features, 1):
    print(f"{i}. {feature}")

print("\nFinal honest ROC-AUC:", round(honest_auc, 4))
print("Final honest accuracy:", round(honest_accuracy, 4))

Final features after leakage removal:
1. feb_gsc_impressions
2. feb_gsc_clicks
3. feb_gsc_avg_position
4. feb_organic_sessions
5. feb_pageviews

Final honest ROC-AUC: 0.8697
Final honest accuracy: 0.825


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

One limitation is uneven data availability across clients. In the February window, only rows with `gsc_data_available IS TRUE` contain measured GSC information, so some content items have missing GSC features. This means the feature history is not equally complete for every content item, and results may not generalize equally across clients with different measurement coverage.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Check the data-availability limitation

limit_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS gsc_unavailable_rows
FROM {FEB}
""").df()

display(limit_check)

available_pct = (
    limit_check.loc[0, "gsc_available_rows"]
    / limit_check.loc[0, "total_rows"]
) * 100

print(f"GSC availability: {available_pct:.2f}%")
print(
    f"GSC unavailable: "
    f"{100 - available_pct:.2f}%"
)

,total_rows,gsc_available_rows,gsc_unavailable_rows
0,7355108,2621783,4733325


GSC availability: 35.65%
GSC unavailable: 64.35%


Self-check

☑ Every section above is filled — markdown thinking AND the code that backs it
☑ The notebook runs top to bottom with no errors
☑ No client names, URLs, or private queries anywhere
☑ My claims use careful words: observed, measured, directional, decision-support
☑ Committed to my repo under work/notebooks/ — then submit my repo URL on the card